In [ ]:
# 01_extract_speech_features

In [ ]:
import os
import pandas as pd


# Route setting
base_dir = "/Users/wangtong/Desktop/26 Spring/STAT 605/group_project"
data_dir = os.path.join(base_dir, "download/outputs/merged")
output_dir = os.path.join(base_dir, "analysis/speech_analysis/outputs")

os.makedirs(output_dir, exist_ok=True)


# Read data
msgs = pd.read_parquet(os.path.join(data_dir, "public_messages.parquet"))
players = pd.read_parquet(os.path.join(data_dir, "players.parquet"))
games = pd.read_parquet(os.path.join(data_dir, "games.parquet"))

print("messages:", msgs.shape)
print("players:", players.shape)
print("games:", games.shape)

print("\nColumns in public_messages:")
print(msgs.columns)


# player-level speech features
# All speak
speech_total = (
    msgs.groupby(["game_id", "speaker_id"])
    .agg(
        n_messages=("text", "size"),
        total_text_len=("text_len", "sum"),
        avg_text_len=("text_len", "mean")
    )
    .reset_index()
    .rename(columns={"speaker_id": "player_id"})
)

# Speech on the first day
speech_day1 = (
    msgs[msgs["day"] == 1]
    .groupby(["game_id", "speaker_id"])
    .agg(
        first_day_messages=("text", "size"),
        first_day_text_len=("text_len", "sum")
    )
    .reset_index()
    .rename(columns={"speaker_id": "player_id"})
)

# Merge into players
speech_features_by_player = (
    players.merge(speech_total, on=["game_id", "player_id"], how="left")
           .merge(speech_day1, on=["game_id", "player_id"], how="left")
)

# Fill in 0 if missing
fill_cols = [
    "n_messages",
    "total_text_len",
    "avg_text_len",
    "first_day_messages",
    "first_day_text_len"
]

for col in fill_cols:
    speech_features_by_player[col] = speech_features_by_player[col].fillna(0)

# Merge the game-level information
speech_features_by_player = speech_features_by_player.merge(
    games[["game_id", "winner_team", "last_day", "n_players", "end_reason"]],
    on="game_id",
    how="left"
)

print("\nSpeech features by player:")
print(speech_features_by_player.head())

speech_features_by_player.to_csv(
    os.path.join(output_dir, "speech_features_by_player.csv"),
    index=False
)


# 4. game-level speech features
game_speech_summary = (
    speech_features_by_player.groupby("game_id")
    .agg(
        total_messages_game=("n_messages", "sum"),
        avg_messages_per_player=("n_messages", "mean"),
        avg_message_length_game=("avg_text_len", "mean"),
        max_messages_one_player=("n_messages", "max")
    )
    .reset_index()
)

game_speech_summary = game_speech_summary.merge(
    games[["game_id", "winner_team", "last_day", "n_players", "end_reason"]],
    on="game_id",
    how="left"
)

print("\nSpeech features by game:")
print(game_speech_summary.head())

game_speech_summary.to_csv(
    os.path.join(output_dir, "speech_features_by_game.csv"),
    index=False
)

print("\nDone. Files saved to:")
print(output_dir)